In [1]:
# 電子透かしを音声に埋め込む
# 埋め込んだ音声をvcにかける
# 抽出モデルを使って透かしを取り出す

In [2]:
import warnings
import torch
import os
import yaml
warnings.filterwarnings("ignore")
from modules.commons import *
from losses import *

import torchaudio
import librosa

import watermark_hparams as hp

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
def load_model(emb_ckpt_path, extract_ckpt_path, config_path):
    emb_ckpt_path = emb_ckpt_path      # 埋め込みモデルのチェックポイントパス
    extract_ckpt_path = extract_ckpt_path  # 抽出モデルのチェックポイントパス
    config_path = config_path
    config = yaml.safe_load(open(config_path))
    model_params = recursive_munch(config['model_params'])
    watemark_model = build_model(model_params, 'watermarking')
    extracter = build_model(model_params, 'extracter')

    emb_ckpt_params = torch.load(emb_ckpt_path)
    emb_ckpt_params = emb_ckpt_params['net'] if 'net' in emb_ckpt_params else emb_ckpt_params  # adapt to format of self-trained checkpoints

    for key in emb_ckpt_params:
        watemark_model[key].load_state_dict(emb_ckpt_params[key])

    _ = [watemark_model[key].eval() for key in watemark_model]
    _ = [watemark_model[key].to(device) for key in watemark_model]

    extract_ckpt_params = torch.load(extract_ckpt_path)
    extract_ckpt_params = extract_ckpt_params['net'] if 'net' in extract_ckpt_params else extract_ckpt_params  # adapt to format of self-trained checkpoints

    for key in extract_ckpt_params:
        extracter[key].load_state_dict(extract_ckpt_params[key])
    
    _ = [extracter[key].eval() for key in extracter]
    _ = [extracter[key].to(device) for key in extracter]

    return watemark_model, extracter

In [6]:
# 透かしの埋め込みを実行
emb_ckpt_path = "/workspace/checkpoints/log1/watermark_model_epoch_5_iter_221735.pth"
extract_ckpt_path = "/workspace/checkpoints/log1/extracter_model_epoch_5_iter_221735.pth"
config_path = "/home/FAcodecWatermark/configs/config.yml"
watermark_model, extracter = load_model(emb_ckpt_path, extract_ckpt_path, config_path)

FileNotFoundError: [Errno 2] No such file or directory: '/workspace/checkpoints/log1/watermark_model_epoch_5_iter_221735.pth'